# 1. Download the dataset

dataset : "Amazon Reviews 2023 (All_Beauty)" directly from the official McAuley (UCSD) Lab


In [1]:
import urllib.request
import json
import pandas as pd
from pathlib import Path

# Initialise directory paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures"
OUT_DIR = PROJECT_ROOT / "output"
for d in (DATA_DIR, FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

# Dowload and load raw datasets
REVIEWS_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/All_Beauty.jsonl"
META_URL    = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_All_Beauty.jsonl"

reviews_path = DATA_DIR / "All_Beauty.jsonl"
meta_path    = DATA_DIR / "meta_All_Beauty.jsonl"

# Downloading datasets : "All_Beauty" & "meta_All_Beauty"
def download_if_needed(url, dest):
    if dest.exists():
        print(f"  Already downloaded: {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"  Downloading {dest.name} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"  Done. Size: {dest.stat().st_size / 1e6:.1f} MB")

print("Step 1: Downloading raw files")
download_if_needed(REVIEWS_URL, reviews_path)
download_if_needed(META_URL, meta_path)

# Reading the JSONL files line by line into a DataFrame
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

print("\nStep 2: Loading reviews into a DataFrame")
df_reviews = load_jsonl(reviews_path)
print(f"  Reviews loaded: {len(df_reviews):,} rows, {df_reviews.shape[1]} columns")
print(f"  Columns: {list(df_reviews.columns)}")

print("\nStep 3: Loading metadata into a DataFrame")
df_meta = load_jsonl(meta_path)
print(f"  Metadata loaded: {len(df_meta):,} rows, {df_meta.shape[1]} columns")

# Save to local drive (so that future downlaods are faster)
print("\nStep 4: Saving as parquet")
df_reviews.to_parquet(DATA_DIR / "reviews_raw.parquet", index=False)
df_meta.to_parquet(DATA_DIR / "meta_raw.parquet", index=False)
print("  Saved: reviews_raw.parquet, meta_raw.parquet")

# Quick preview of the Reviews dataset
print("\nStep 5: Quick preview of the Reviews dataset")
print(df_reviews.head(10).to_string())
print(f"\n  helpful_vote stats:")
print(df_reviews["helpful_vote"].describe())
print(f"\n  rating distribution:")
print(df_reviews["rating"].value_counts().sort_index())

Project root: /Users/brg_elise/Desktop/Dissertation_Analysis
Step 1: Downloading raw files
  Already downloaded: All_Beauty.jsonl (326.6 MB)
  Already downloaded: meta_All_Beauty.jsonl (213.0 MB)

Step 2: Loading reviews into a DataFrame
  Reviews loaded: 701,528 rows, 10 columns
  Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Step 3: Loading metadata into a DataFrame
  Metadata loaded: 112,590 rows, 14 columns

Step 4: Saving as parquet
  Saved: reviews_raw.parquet, meta_raw.parquet

Step 5: Quick preview of the Reviews dataset
   rating                                      title                                                                                                                                                                                                                                                                                                                                       

# 2. Clean & Merge reviews and metadata

Before analysing each dataset (to identify missing or incorrect values), we need to examine our data tables one by one. This will enable us to identify empty variables (those containing no values and which are therefore of little use to our research), as well as those containing outliers and/or incorrect values.

## 2.1. Load the two saved data tables

In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.home() / "Desktop" / "Dissertation_Analysis"
DATA_DIR = PROJECT_ROOT / "data"

print("Step 1: Loading parquet files")
df_reviews = pd.read_parquet(DATA_DIR / "reviews_raw.parquet")
df_meta = pd.read_parquet(DATA_DIR / "meta_raw.parquet")
print(f"  Reviews: {len(df_reviews):,} rows | Meta: {len(df_meta):,} rows")

Step 1: Loading parquet files
  Reviews: 701,528 rows | Meta: 112,590 rows


## 2.2. Analysis of the ‘Reviews’ dataset

In [3]:
# Identify structure and verify data distribution

print("\n--- ANALYSIS OF DF_REVIEWS---")

# Check the structure and data types
print("\n1. General Informations :")
df_reviews.info()

# Calculate percentage of missing values per column (NaN)
missing_percent = (df_reviews.isnull().sum() / len(df_reviews)) * 100
print("\n2. Percentage of missing values per column :")
print(missing_percent.sort_values(ascending=False))

# Preview a sample of data (around twenty)
print("\n3. Preview of the 'df_reviews' dataset :")
pd.set_option('display.max_columns', None)  
pd.set_option('display.max_rows', None)      
display(df_reviews.head(20))


--- ANALYSIS OF DF_REVIEWS---

1. General Informations :
<class 'pandas.DataFrame'>
RangeIndex: 701528 entries, 0 to 701527
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             701528 non-null  float64
 1   title              701528 non-null  str    
 2   text               701528 non-null  str    
 3   images             701528 non-null  object 
 4   asin               701528 non-null  str    
 5   parent_asin        701528 non-null  str    
 6   user_id            701528 non-null  str    
 7   timestamp          701528 non-null  int64  
 8   helpful_vote       701528 non-null  int64  
 9   verified_purchase  701528 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 211.3+ MB

2. Percentage of missing values per column :
rating               0.0
title                0.0
text                 0.0
images               0.0
asin                 0.0
pa

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True
5,4.0,Pretty Color,The polish was quiet thick and did not apply s...,[{'small_image_url': 'https://images-na.ssl-im...,B00R8DXL44,B00R8DXL44,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,1598567408138,0,True
6,5.0,Handy,Great for many tasks. I purchased these for m...,[],B099DRHW5V,B099DRHW5V,AHREXOGQPZDA6354MHH4ETSF3MCQ,1631885519443,0,True
7,3.0,Meh,These were lightweight and soft but much too s...,[{'small_image_url': 'https://m.media-amazon.c...,B088SZDGXG,B08BBQ29N5,AEYORY2AVPMCPDV57CE337YU5LXA,1634275259292,0,True
8,5.0,Great for at home use and so easy to use!,This is perfect for my between salon visits. I...,[],B08P2DZB4X,B08P2DZB4X,AFSKPY37N3C43SOI5IEXEK5JSIYA,1627391044559,0,False
9,5.0,Nice shampoo for the money,I get Keratin treatments at the salon at least...,[],B086QY6T7N,B086QY6T7N,AFSKPY37N3C43SOI5IEXEK5JSIYA,1626614511145,0,False


No empty column found. Howevern preview shows '[]' values.

These do not add informations. Let's quantify them.

In [4]:
# Identify and count '[]' values per columnt
empty_lists_count = (df_reviews.astype(str) == '[]').sum()

empty_lists_percent = (empty_lists_count / len(df_reviews)) * 100

df_empty_report = pd.DataFrame({
    'Incorrect_Values_[]': empty_lists_count,
    'Percentage_%': empty_lists_percent.round(2)
})

# Show only affected columns, sorting descending
print("\n--- ANALYSIS OF INCORRECT VALUES '[]' ---")
print(df_empty_report[df_empty_report['Incorrect_Values_[]'] > 0].sort_values(by='Incorrect_Values_[]', ascending=False))


--- ANALYSIS OF INCORRECT VALUES '[]' ---
        Incorrect_Values_[]  Percentage_%
images               641844         91.49


The 'images' column contains 641 844 '[]' values -> 91.4 %.
Convert to a binary variable (0 to 1) for usability.

In [5]:
# Convert “images” to a binary variable named 'has_image' (1 if image exists, 0 otherwise)
df_reviews['has_image'] = (df_reviews['images'].astype(str) != '[]').astype(int)

# Verify 'has_image' distribution
print(df_reviews['has_image'].value_counts(normalize=True) * 100)

# Drop redundant 'images' column
df_reviews = df_reviews.drop(columns=['images'])

has_image
0    91.492285
1     8.507715
Name: proportion, dtype: float64


## 2.3. Analysis of the ‘Meta’ dataset

In [6]:
print("\n--- ANALYSIS OF DF_META ---")

# Check the structure and data types
print("\n1. General Informations :")
df_meta.info()

# Calculate percentage of missing values per column (NaN)
missing_percent = (df_meta.isnull().sum() / len(df_meta)) * 100
print("\n2. Percentage of missing values per column :")
print(missing_percent.sort_values(ascending=False))

# Preview a sample of data (around twenty)
print("\n3. Preview of the first 20 rows :")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
display(df_meta.head(20)) 


--- ANALYSIS OF DF_META ---

1. General Informations :
<class 'pandas.DataFrame'>
RangeIndex: 112590 entries, 0 to 112589
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    112590 non-null  str    
 1   title            112590 non-null  str    
 2   average_rating   112590 non-null  float64
 3   rating_number    112590 non-null  int64  
 4   features         112590 non-null  object 
 5   description      112590 non-null  object 
 6   price            17704 non-null   float64
 7   images           112590 non-null  object 
 8   videos           112590 non-null  object 
 9   store            101259 non-null  str    
 10  categories       112590 non-null  object 
 11  details          112590 non-null  object 
 12  parent_asin      112590 non-null  str    
 13  bought_together  0 non-null       object 
dtypes: float64(2), int64(1), object(7), str(4)
memory usage: 27.3+ MB

2. Percentage of mis

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,None
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Package Dimensions': None, 'UPC': '653801351...",B076WQZGPM,None
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],"{'Package Dimensions': None, 'UPC': None, 'Ite...",B000B658RI,None
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cherioll,[],{'Package Dimensions': '8.43 x 5.91 x 0.87 inc...,B088FKY3VD,None
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Precision,[],"{'Package Dimensions': None, 'UPC': '644287689...",B07NGFDN6G,None
5,All Beauty,Lurrose 100Pcs Full Cover Fake Toenails Artifi...,3.7,35,[The false toenails are durable with perfect l...,"[Description, The false toenails are durable w...",6.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Lurrose,[],"{'Package Dimensions': None, 'UPC': '799768026...",B07G9GWFSM,None
6,All Beauty,Stain Bonnet For Baby Bonnet Silk Sleep Cap Fo...,4.1,50,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Edoneery,[],{'Package Dimensions': '10 x 5.79 x 0.75 inche...,B08XZ97HFY,None
7,All Beauty,50 Pieces False Eyelash Packaging Box Empty Ey...,3.8,32,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Maitys,[],{'Package Dimensions': '14.49 x 11.26 x 2.36 i...,B08DNQTTQK,None
8,All Beauty,Gold extatic Musk EDT 90ml,3.7,2,[Extatic Balmain Gold Musk By Balmain Edt Spra...,[Edt spray 3 oz design house: balmain],86.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],Balmain,[],"{'Package Dimensions': None, 'UPC': None, 'Ite...",B01ERJEGS6,None
9,All Beauty,4 Pieces Satin Bonnet Adjustable Sleep Cap Dou...,4.3,66,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Geyoga,[],{'Package Dimensions': '12.49 x 9.97 x 1.46 in...,B08P7LXKP7,None


Only the columns : 'price', 'store' & 'bought_together' contain missing values.

In [7]:
# Estimate the real number and percentage of '[]' values in the metadata DataFrame

## Identify and count '[]' values per column
empty_lists_count = (df_meta.astype(str) == '[]').sum()

## Calculate percentage relative to the total number of rows
empty_lists_percent = (empty_lists_count / len(df_meta)) * 100

## Create a clean summary table
df_empty_report = pd.DataFrame({
    'Valeurs_Vides_[]': empty_lists_count,
    'Pourcentage_%': empty_lists_percent.round(2)
})

## Show only affected columns, sorting descending
print("\n--- ANALYSIS OF INCORRECT VALUES '[]' ---")
print(df_empty_report[df_empty_report['Valeurs_Vides_[]'] > 0].sort_values(by='Valeurs_Vides_[]', ascending=False))


--- ANALYSIS OF INCORRECT VALUES '[]' ---
             Valeurs_Vides_[]  Pourcentage_%
categories             112590         100.00
videos                  96708          85.89
features                95213          84.57
description             93428          82.98


Based on the previous steps, drop irrelevant/empty columns:
- "bought_together" (100% empty),
- "categories" (100% '[]'),
- "videos", "features", "description" contain a lot of '[]' values (between 80 and 85 %)

In [8]:
# --- Drop empty metadata columns ---
print("\nStep 2: Dropping empty columns in metadata")
cols_to_drop_meta = ["bought_together", "categories", "videos", "features", "description"]
cols_present = [c for c in cols_to_drop_meta if c in df_meta.columns]
df_meta = df_meta.drop(columns=cols_present)
print(f"  Dropped: {cols_present}")
print(f"  Meta columns remaining: {list(df_meta.columns)}")

# Preview after cleaning
print("\n3. Preview of the first 20 rows :")
pd.set_option('display.max_columns', None)  # Afficher toutes les colonnes
pd.set_option('display.max_rows', None)     # Afficher toutes les lignes        # Ajuster la largeur d'affichage
display(df_meta.head(20))

# Check for duplicate review texts
n_dupes = df_reviews["text"].duplicated().sum()
print(f"Duplicate review texts: {n_dupes:,} ({n_dupes/len(df_reviews)*100:.2f}%)")


Step 2: Dropping empty columns in metadata
  Dropped: ['bought_together', 'categories', 'videos', 'features', 'description']
  Meta columns remaining: ['main_category', 'title', 'average_rating', 'rating_number', 'price', 'images', 'store', 'details', 'parent_asin']

3. Preview of the first 20 rows :


,main_category,title,average_rating,rating_number,price,images,store,details,parent_asin
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Howard Products,{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Yes To,"{'Package Dimensions': None, 'UPC': '653801351...",B076WQZGPM
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Levine Health Products,"{'Package Dimensions': None, 'UPC': None, 'Ite...",B000B658RI
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Cherioll,{'Package Dimensions': '8.43 x 5.91 x 0.87 inc...,B088FKY3VD
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Precision,"{'Package Dimensions': None, 'UPC': '644287689...",B07NGFDN6G
5,All Beauty,Lurrose 100Pcs Full Cover Fake Toenails Artifi...,3.7,35,6.99,[{'thumb': 'https://m.media-amazon.com/images/...,Lurrose,"{'Package Dimensions': None, 'UPC': '799768026...",B07G9GWFSM
6,All Beauty,Stain Bonnet For Baby Bonnet Silk Sleep Cap Fo...,4.1,50,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Edoneery,{'Package Dimensions': '10 x 5.79 x 0.75 inche...,B08XZ97HFY
7,All Beauty,50 Pieces False Eyelash Packaging Box Empty Ey...,3.8,32,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Maitys,{'Package Dimensions': '14.49 x 11.26 x 2.36 i...,B08DNQTTQK
8,All Beauty,Gold extatic Musk EDT 90ml,3.7,2,86.95,[{'thumb': 'https://m.media-amazon.com/images/...,Balmain,"{'Package Dimensions': None, 'UPC': None, 'Ite...",B01ERJEGS6
9,All Beauty,4 Pieces Satin Bonnet Adjustable Sleep Cap Dou...,4.3,66,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Geyoga,{'Package Dimensions': '12.49 x 9.97 x 1.46 in...,B08P7LXKP7


Duplicate review texts: 57,899 (8.25%)


This Datalake containes **8.25 %** of duplicate review texts.

These are overwhelmingly short, generic phrases (e.g. "Love it", "As described", "Great product") that many users write independently, rather than bot - or spam-generated duplicates.
Each entry remains a distict review, with its own rating and helpful_vote, so all rows are kept.  

In [9]:
# --- Merge ---
# Merge on 'parent_asin'. Use left join to retain all reviews.
print("\nStep 3: Merging reviews with metadata on parent_asin")
df = df_reviews.merge(
    df_meta,
    on="parent_asin",
    how="left",
    suffixes=("_review", "_product"),
)
print(f"  Merged dataset: {len(df):,} rows, {df.shape[1]} columns")
print(f"  Columns: {list(df.columns)}")

# Preview after merging
print("\n3. Preview of the first 20 rows :")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
display(df.head(20)) 


Step 3: Merging reviews with metadata on parent_asin
  Merged dataset: 701,528 rows, 18 columns
  Columns: ['rating', 'title_review', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'has_image', 'main_category', 'title_product', 'average_rating', 'rating_number', 'price', 'images', 'store', 'details']

3. Preview of the first 20 rows :


,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,has_image,main_category,title_product,average_rating,rating_number,price,images,store,details
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True,0,All Beauty,Herbivore - Natural Sea Mist Texturizing Salt ...,4.3,384,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,HERBIVORE,{'Package Dimensions': '9.57 x 3.27 x 3.15 inc...
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True,0,All Beauty,All Natural Vegan Dry Shampoo Powder - Eco Fri...,4.0,56,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Two Goats Apothecary,{'Package Dimensions': '6.6 x 4.2 x 1.5 inches...
2,5.0,Yes!,"Smells good, feels great!",B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True,0,All Beauty,New Road Beauty - Creamsicle - Variety 3 Pack ...,4.4,699,21.98,[{'thumb': 'https://m.media-amazon.com/images/...,New Road Beauty,{'Package Dimensions': '10.5 x 6.4 x 1.6 inche...
3,1.0,Synthetic feeling,Felt synthetic,B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True,0,All Beauty,muaowig Ombre Body Wave Bundles 1B Grey Human ...,1.0,1,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,muaowig,{'Package Dimensions': '13.94 x 10.43 x 2.32 i...
4,5.0,A+,Love it,B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True,0,All Beauty,Yinhua Electric Nail Drill Kit Portable Profes...,3.5,20,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Yinhua,{'Package Dimensions': '8.5 x 3.82 x 2.24 inch...
5,4.0,Pretty Color,The polish was quiet thick and did not apply s...,B00R8DXL44,B00R8DXL44,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,1598567408138,0,True,1,All Beauty,"China Glaze Nail Polish, Wanderlust 1381",3.8,32,7.10,[{'thumb': 'https://m.media-amazon.com/images/...,China Glaze,"{'Package Dimensions': None, 'UPC': '019965823..."
6,5.0,Handy,Great for many tasks. I purchased these for m...,B099DRHW5V,B099DRHW5V,AHREXOGQPZDA6354MHH4ETSF3MCQ,1631885519443,0,True,0,All Beauty,"Disposable Facial Cotton Tissue, 100PCS Cotton...",3.5,4,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,AYQNMHR,{'Package Dimensions': '8.58 x 4.37 x 3.27 inc...
7,3.0,Meh,These were lightweight and soft but much too s...,B088SZDGXG,B08BBQ29N5,AEYORY2AVPMCPDV57CE337YU5LXA,1634275259292,0,True,1,All Beauty,Niseyo new Faux Locs 24 Inch Crochet Hair 6 Pa...,4.3,62,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Niseyo,"{'Package Dimensions': None, 'UPC': '782931053..."
8,5.0,Great for at home use and so easy to use!,This is perfect for my between salon visits. I...,B08P2DZB4X,B08P2DZB4X,AFSKPY37N3C43SOI5IEXEK5JSIYA,1627391044559,0,False,0,All Beauty,NIRA Skincare Laser & Serum Bundle - Includes ...,3.8,109,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Nira,"{'Package Dimensions': None, 'UPC': None, 'Ite..."
9,5.0,Nice shampoo for the money,I get Keratin treatments at the salon at least...,B086QY6T7N,B086QY6T7N,AFSKPY37N3C43SOI5IEXEK5JSIYA,1626614511145,0,False,0,All Beauty,Caroline Keller Keratin Shampoo for dry and da...,3.5,12,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Caroline Keller,"{'Package Dimensions': None, 'UPC': None, 'Ite..."


In [10]:
# --- 4. Compute 'text_length' ---
print("\nStep 4: Computing text_length (word count)")
df["text"] = df["text"].fillna("")
df["text_length"] = df["text"].str.split().str.len()
print(f"  text_length stats: mean={df['text_length'].mean():.1f}, "
      f"median={df['text_length'].median():.0f}, "
      f"max={df['text_length'].max()}")


Step 4: Computing text_length (word count)
  text_length stats: mean=32.8, median=19, max=2585


In this dataset, the reviews analysed average **33 words**.

In [11]:
# --- Filter invalid rows (with empty text or missing rating) ---
print("\nStep 5: Filtering invalid rows")
n_before = len(df)
df = df[df["text"].str.strip() != ""]
df = df[df["rating"].notna()]
df = df.reset_index(drop=True)
print(f"  Removed {n_before - len(df):,} rows (empty text or missing rating)")
print(f"  Final dataset: {len(df):,} rows")


Step 5: Filtering invalid rows
  Removed 720 rows (empty text or missing rating)
  Final dataset: 700,808 rows


After cleaning, we have moved from a dataset containing 701 528 rows to a database containing **700 808** rows.

In [12]:
# --- Save final dataset ---
output_path = DATA_DIR / "reviews_clean.parquet"
df.to_parquet(output_path, index=False)
print(f"\nStep 6: Saved to {output_path}")
print(f"  File size: {output_path.stat().st_size / 1e6:.1f} MB")

# --- 7. Print a final preview ---
print("\nFinal dataset preview:")
print(df[["rating", "text_length", "helpful_vote"]].describe())
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print("\nOverview of the first 20 observations (all columns) :")
display(df.head(20))


Step 6: Saved to /Users/brg_elise/Desktop/Dissertation_Analysis/data/reviews_clean.parquet
  File size: 322.4 MB

Final dataset preview:
              rating    text_length   helpful_vote
count  700808.000000  700808.000000  700808.000000
mean        3.960196      32.784367       0.923804
std         1.494365      45.984890       5.473788
min         1.000000       1.000000       0.000000
25%         3.000000       8.000000       0.000000
50%         5.000000      19.000000       0.000000
75%         5.000000      40.000000       1.000000
max         5.000000    2585.000000     646.000000

Rating distribution:
rating
1.0    101948
2.0     43009
3.0     56283
4.0     79318
5.0    420250
Name: count, dtype: int64

Overview of the first 20 observations (all columns) :


,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,has_image,main_category,title_product,average_rating,rating_number,price,images,store,details,text_length
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True,0,All Beauty,Herbivore - Natural Sea Mist Texturizing Salt ...,4.3,384,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,HERBIVORE,{'Package Dimensions': '9.57 x 3.27 x 3.15 inc...,61
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True,0,All Beauty,All Natural Vegan Dry Shampoo Powder - Eco Fri...,4.0,56,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Two Goats Apothecary,{'Package Dimensions': '6.6 x 4.2 x 1.5 inches...,47
2,5.0,Yes!,"Smells good, feels great!",B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True,0,All Beauty,New Road Beauty - Creamsicle - Variety 3 Pack ...,4.4,699,21.98,[{'thumb': 'https://m.media-amazon.com/images/...,New Road Beauty,{'Package Dimensions': '10.5 x 6.4 x 1.6 inche...,4
3,1.0,Synthetic feeling,Felt synthetic,B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True,0,All Beauty,muaowig Ombre Body Wave Bundles 1B Grey Human ...,1.0,1,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,muaowig,{'Package Dimensions': '13.94 x 10.43 x 2.32 i...,2
4,5.0,A+,Love it,B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True,0,All Beauty,Yinhua Electric Nail Drill Kit Portable Profes...,3.5,20,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Yinhua,{'Package Dimensions': '8.5 x 3.82 x 2.24 inch...,2
5,4.0,Pretty Color,The polish was quiet thick and did not apply s...,B00R8DXL44,B00R8DXL44,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,1598567408138,0,True,1,All Beauty,"China Glaze Nail Polish, Wanderlust 1381",3.8,32,7.10,[{'thumb': 'https://m.media-amazon.com/images/...,China Glaze,"{'Package Dimensions': None, 'UPC': '019965823...",24
6,5.0,Handy,Great for many tasks. I purchased these for m...,B099DRHW5V,B099DRHW5V,AHREXOGQPZDA6354MHH4ETSF3MCQ,1631885519443,0,True,0,All Beauty,"Disposable Facial Cotton Tissue, 100PCS Cotton...",3.5,4,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,AYQNMHR,{'Package Dimensions': '8.58 x 4.37 x 3.27 inc...,22
7,3.0,Meh,These were lightweight and soft but much too s...,B088SZDGXG,B08BBQ29N5,AEYORY2AVPMCPDV57CE337YU5LXA,1634275259292,0,True,1,All Beauty,Niseyo new Faux Locs 24 Inch Crochet Hair 6 Pa...,4.3,62,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Niseyo,"{'Package Dimensions': None, 'UPC': '782931053...",32
8,5.0,Great for at home use and so easy to use!,This is perfect for my between salon visits. I...,B08P2DZB4X,B08P2DZB4X,AFSKPY37N3C43SOI5IEXEK5JSIYA,1627391044559,0,False,0,All Beauty,NIRA Skincare Laser & Serum Bundle - Includes ...,3.8,109,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Nira,"{'Package Dimensions': None, 'UPC': None, 'Ite...",72
9,5.0,Nice shampoo for the money,I get Keratin treatments at the salon at least...,B086QY6T7N,B086QY6T7N,AFSKPY37N3C43SOI5IEXEK5JSIYA,1626614511145,0,False,0,All Beauty,Caroline Keller Keratin Shampoo for dry and da...,3.5,12,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,Caroline Keller,"{'Package Dimensions': None, 'UPC': None, 'Ite...",89
